# Tutorial: Supporting STREAM datasets

STREAM combines continuous water-quality observations with grab samples, streamflow, historical meteorology, annual land cover, annual anthropogenic variables, static catchment attributes, and watershed geometries. This tutorial shows how to discover a viable site and retrieve each supporting dataset without downloading the complete resource.

## 1. Connect to HydroShare

In [ ]:
import streamwq as stream
from streamwq.helpers.pandas import resample_timeseries

## 2. Find a site with the required datasets

Use metadata availability to find candidate sites for water temperature, streamflow, and dissolved organic carbon grab samples. `stream.get_sites()` combines different filters (water-quality variables, streamflow, and grab-sample availability) and accepts alternatives within a single variable filter. Discovery identifies candidates from metadata; it does not guarantee observations for every dataset, date range, or quality filter.

In [ ]:
candidate_sites = stream.get_sites(
    variables="WTemp_C",
    streamflow=True,
    grab_variables="DOC_mgL",
)

print(f"Found {len(candidate_sites):,} candidate sites")
candidate_metadata = stream.get_metadata(
    site_ids=candidate_sites,
    columns=["STREAM_ID", "site_name", "state_name"],
)
candidate_metadata.slice(0, 10)

Select the first matching site so this tutorial adapts to future resource versions.

In [ ]:
example_site = candidate_sites.column("STREAM_ID")[0].as_py()
print(f"Using {example_site}")

## 3. Read grab-sample measurements

Grab samples are discrete laboratory or field measurements. They do not have paired source-quality flags in STREAM.

In [ ]:
grab_samples = stream.get_data(
    "grab_samples",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "DOC_mgL"],
)

print(f"Retrieved {len(grab_samples):,} grab samples")
if grab_samples.num_rows == 0:
    print("No grab samples were returned for this candidate site.")
grab_samples.slice(0, 10)

Grab samples should not automatically be joined to hourly sensor observations using exact timestamps. The appropriate matching window depends on the research question and sampling protocol.

## 4. Read historical meteorology

Meteorology is watershed-averaged and uses the `DateTime` column. Air temperature is provided in kelvin, and precipitation is reported in kilograms per square meter, equivalent to millimeters of water.

In [ ]:
meteorology = stream.get_data(
    "meteorology",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "APCP_surface", "TMP_2maboveground"],
    start="2020-01-01",
    end="2020-12-31 23:59:59",
).to_pandas()

if meteorology.empty:
    raise ValueError("No meteorology observations matched this candidate site and date range")

Different variables may require different aggregation statistics. Sum precipitation and average temperature in separate calls.

In [ ]:
daily_precipitation = resample_timeseries(
    meteorology,
    value_columns="APCP_surface",
    frequency="daily",
    statistic="sum",
)

daily_temperature = resample_timeseries(
    meteorology,
    value_columns="TMP_2maboveground",
    frequency="daily",
    statistic="mean",
)
daily_temperature["temperature_C"] = (
    daily_temperature["TMP_2maboveground"] - 273.15
)

daily_temperature.head()

## 5. Read annual land-cover fractions

Land-cover values are watershed fractions from 0 to 1. Passing `variables=None` retrieves all 16 classes; this example requests only four.

In [ ]:
land_cover = stream.get_data(
    "land_cover",
    site_ids=example_site,
    columns=[
        "STREAM_ID", "DateTime", "Developed Open Space",
        "Deciduous Forest", "Cultivated Crops", "Woody Wetlands",
    ],
    start="2015-01-01",
    end="2020-12-31",
)

land_cover

Because only four classes were selected, their values are not expected to sum to 1. The sum-to-one property applies across all 16 classes for a site and year.

## 6. Read annual anthropogenic variables

Anthropogenic variables have different units, source periods, and temporal coverage. Review the STREAM variable documentation before comparing them. Passing `variables=None` retrieves all available variables.

In [ ]:
anthropogenic = stream.get_data(
    "anthropogenic",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "salt_pound", "cl_tw", "nfer", "irrig"],
    start="2000-01-01",
    end="2020-12-31",
)

anthropogenic.slice(0, 10)

## 7. Read static catchment attributes

Static attributes describe relatively fixed watershed characteristics. Select only the columns required by the analysis.

In [ ]:
static_attributes = stream.get_data(
    "static_catchment_characteristics",
    site_ids=example_site,
    columns=["STREAM_ID", "stream_order", "ele_mt_sav", "for_pc_sse", "urb_pc_sse"],
)

static_attributes

## 8. Read the watershed geometry

Watersheds are returned as a GeoPandas GeoDataFrame in their source coordinate reference system. STREAM documents the polygons in CONUS Albers Equal Area (`EPSG:5070`), which is appropriate for area calculations.

In [ ]:
watershed_table = stream.get_watersheds(site_ids=example_site)
watershed = stream.helpers.to_geopandas(watershed_table)

print("CRS:", watershed.crs)
watershed[["STREAM_ID", "COMID", "area_km2"]]

In [ ]:
# Reproject explicitly before displaying in a longitude/latitude web map.
watershed_web = watershed.to_crs("EPSG:4326")
watershed_web.plot(figsize=(7, 7), edgecolor="navy", facecolor="lightblue")

## Interpretation notes

- Supporting datasets have different temporal resolutions and source periods.
- Do not assume that every variable is available for every year.
- Use variable-specific aggregation statistics.
- Preserve units and review source documentation before derived calculations.
- Use `STREAM_ID` to connect datasets, but align temporal datasets at scientifically appropriate resolutions.
- Review watershed delineation and hydrologic-impact flags in metadata before catchment-scale analysis.